## Klima Klassifikator

Dieses Notebook in englischer Sprache erläutert die Anwendung des Klassifikators.

### Install Requirements

In order to use this classifier, please install the requirements from the provided file. It is recommended to install in a fresh virtual environment. Use the following command in your command line interface while in this directory:

> pip install -r requirements.txt

In [ ]:
import pandas as pd
import xgboost as xgb

from utils import read_txt_files, add_features, create_embeddings, create_x

### Data Input

Choose **one** of these data input methods. Select your method by setting it to True and the other to False. Input the path as follows:

- **Option 1 csv file:** Input the full or relative path to the .csv file containing your data. It should contain a column called 'text'. Example input: `path = 'data/texts.csv'`

OR

- **Option 2 txt files:** Input the path to a folder containing txt files with the texts you want to classify. Note that *all* txt files in this folder will be read and there will be no preprocessing or splitting of the texts. Long texts may not be handled properly and will be truncated during classification. Example input: `path = 'data/texts/'`

The classifier was trained with the year of the text's authorship as a feature. If inputting a csv, you may wish to add a column 'year' with the respective year. Otherwise an empty column is added during feature addition.

In [ ]:
OPTION_1 = False
OPTION_2 = True
path = 'texts'

In [ ]:
if OPTION_1 and not OPTION_2:
    data = pd.read_csv(path, index_col=0)
elif OPTION_2 and not OPTION_1:
    data = read_txt_files(dir_path=path, encoding='utf-8')
else:
    print("Please select one of the input options as described above.")

The following function adds feature counts to the data and shows what the data looks like after.

In [ ]:
data = add_features(data)
data.head()

#### Embeddings

Embeddings are created to feed text into the classifier. It creates a file called `embeddings.npy`. If you already have a file `embeddings.npy` for the texts you want to classify, **you do not need to** repeat this step, simply skip to the next step.

In [ ]:
texts = data['text'].to_list()
create_embeddings(texts)

#### Classification

Now we will create the input matrix and pass it into the classifier. Results of classification are saved in two files: one saves all texts with the assigned labels and one only those texts labeled 1 or topic-relevant.

In [ ]:
X = create_x(data=data, embeddings_file_path='embeddings.npy')

xgb_model = xgb.XGBClassifier()
xgb_model.load_model('xgb_model.json')
preds = xgb_model.predict(X)    #make predictions

# concatenate data with predicted label and save
preds_df = pd.DataFrame(preds, columns=['pred_y'])

preds_df.reset_index(drop=True, inplace=True)
data.reset_index(drop=True, inplace=True)

result = pd.concat([data, preds_df], axis=1)
result.to_csv('texts_classified.csv')
result[result['class']==1].to_csv('texts_climate.csv')

#### Viewing results

In [ ]:
result.groupby('class').count()

In [ ]:
result.loc[result['class']==0]['text'].iloc[0]